In [9]:
class Problem:
    def __init__(self, initial, goal=None):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def goal_test(self, state):
        if isinstance(self.goal, list):
            return state in self.goal
        return state == self.goal

    def path_cost(self, c, state1, action, state2):
        return c + 1

In [5]:
# Exercise 3.1

class EightPuzzle(Problem):
    def __init__(self, initial, goal=(1, 2, 3, 4, 5, 6, 7, 8, 0)):
        super().__init__(initial, goal)  # Super Parent class ca constructor call krta hai.

    def actions(self, state):
        # Actions define the movement of the blank space (Up, Down, Left, Right) [cite: 51]
        moves = []
        idx = state.index(0)
        if idx not in (0, 1, 2): moves.append('UP')
        if idx not in (6, 7, 8): moves.append('DOWN')
        if idx not in (0, 3, 6): moves.append('LEFT')
        if idx not in (2, 5, 8): moves.append('RIGHT')
        return moves

    def result(self, state, action):
        # The transition model returns the resulting state after sliding [cite: 51]
        new_state = list(state)
        idx = state.index(0)
        if action == 'UP': neighbor = idx - 3
        elif action == 'DOWN': neighbor = idx + 3
        elif action == 'LEFT': neighbor = idx - 1
        elif action == 'RIGHT': neighbor = idx + 1
        
        new_state[idx], new_state[neighbor] = new_state[neighbor], new_state[idx]
        return tuple(new_state)

In [9]:
# Exerscise 3.2

class AnimalProblem(Problem):
    def __init__(self, target_animal):
        # State: Current set of questions answered
        super().__init__(initial=(), goal=target_animal)
        self.tree = {
            "Lion": ("No water", "No fly", "No trunk"),
            "Shark": ("Water",),
            "Eagle": ("No water", "Fly"),
            "Elephant": ("No water", "No fly", "Trunk")
        }

    def actions(self, state):
        # Actions are the possible answers/characteristics to explore
        all_features = ["Water", "No water", "Fly", "No fly", "Trunk", "No trunk"]
        return [f for f in all_features if f not in state]

    def result(self, state, action):
        return state + (action,)

    def goal_test(self, state):
        # Goal test: Does the current state match the target animal's features? [cite: 10]
        if self.goal in self.tree:
            required = self.tree[self.goal]
            return all(feature in state for feature in required)
        return False


In [10]:
# Exercise 3.3

class MissionariesAndCannibals(Problem):
    def __init__(self):
        # State: (Missionaries, Cannibals, Boat_at_Start_Bank) 
        super().__init__(initial=(3, 3, 1), goal=(0, 0, 0))

    def actions(self, state):
        m, c, b = state
        possible_moves = [(1,0), (2,0), (0,1), (0,2), (1,1)]
        valid = []
        for dm, dc in possible_moves:
            # Transition based on boat position 
            new_s = (m-dm, c-dc, 0) if b == 1 else (m+dm, c+dc, 1)
            if self.is_safe(new_s):
                valid.append((dm, dc))
        return valid

    def is_safe(self, state):
        m, c, b = state
        if not (0 <= m <= 3 and 0 <= c <= 3): return False
        if m > 0 and m < c: return False # Start bank safety
        if (3-m) > 0 and (3-m) < (3-c): return False # Dest bank safety
        return True

    def result(self, state, action):
        m, c, b = state
        dm, dc = action
        return (m-dm, c-dc, 0) if b == 1 else (m+dm, c+dc, 1)

In [11]:
# Exercise 3.4

class FarmerProblem(Problem):
    def __init__(self):
        # State: (Farmer, Wolf, Chicken, Grain) 0=Start, 1=Dest [cite: 155]
        super().__init__(initial=(0, 0, 0, 0), goal=(1, 1, 1, 1))

    def actions(self, state):
        f, w, c, g = state
        moves = ['Alone']
        if w == f: moves.append('Wolf')
        if c == f: moves.append('Chicken')
        if g == f: moves.append('Grain')
        
        # Filter actions that leave items in an unsafe state [cite: 157]
        return [a for a in moves if self.is_safe(self.result(state, a))]

    def is_safe(self, state):
        f, w, c, g = state
        if w == c and w != f: return False # Wolf eats Chicken [cite: 157]
        if c == g and c != f: return False # Chicken eats Grain [cite: 157]
        return True

    def result(self, state, action):
        f, w, c, g = list(state)
        nf = 1 - f # Farmer always moves [cite: 156]
        if action == 'Wolf': return (nf, 1-w, c, g)
        if action == 'Chicken': return (nf, w, 1-c, g)
        if action == 'Grain': return (nf, w, c, 1-g)
        return (nf, w, c, g)
        

In [6]:
# BFS - Algorithm for Execution

from collections import deque

def breadth_first_search(problem):
    """Finds the shallowest node in the search tree."""
    node = problem.initial
    if problem.goal_test(node):
        return [node]
    
    # Frontier is a FIFO queue for BFS 
    frontier = deque([[node]])
    explored = set()
    
    while frontier:
        path = frontier.popleft()
        current_state = path[-1]
        
        if current_state not in explored:
            explored.add(current_state)
            
            for action in problem.actions(current_state):
                child = problem.result(current_state, action)
                new_path = list(path)
                new_path.append(child)
                
                if problem.goal_test(child):
                    return new_path
                
                frontier.append(new_path)
    return None

In [ ]:
# for 3.1 
start_state = (1, 2, 3, 4, 5, 6, 7, 0, 8) 

puzzle = EightPuzzle(initial=start_state)
solution = breadth_first_search(puzzle)

print("\n--- 8-Puzzle Solution ---")
for step, state in enumerate(solution):
    print(f"Step {step}: {state[0:3]}\n        {state[3:6]}\n        {state[6:9]}\n")


--- 8-Puzzle Solution ---
Step 0: (1, 2, 3)
        (4, 5, 6)
        (7, 0, 8)

Step 1: (1, 2, 3)
        (4, 5, 6)
        (7, 8, 0)



In [14]:
# for 3.2
target = "Elephant"
animal_problem = AnimalProblem(target)
solution = breadth_first_search(animal_problem)

print(f"--- Path to identify {target} ---")
print(solution)


--- Path to identify Elephant ---
[(), ('No water',), ('No water', 'No fly'), ('No water', 'No fly', 'Trunk')]


In [15]:
# for 3.3

mc_problem = MissionariesAndCannibals()
solution = breadth_first_search(mc_problem)

print("--- Missionaries and Cannibals Solution ---")
for step, state in enumerate(solution):
    print(f"Step {step}: {state}")

--- Missionaries and Cannibals Solution ---
Step 0: (3, 3, 1)
Step 1: (3, 1, 0)
Step 2: (3, 2, 1)
Step 3: (3, 0, 0)
Step 4: (3, 1, 1)
Step 5: (1, 1, 0)
Step 6: (2, 2, 1)
Step 7: (0, 2, 0)
Step 8: (0, 3, 1)
Step 9: (0, 1, 0)
Step 10: (1, 1, 1)
Step 11: (0, 0, 0)


In [16]:
# for 3.4

farmer_problem = FarmerProblem()
solution = breadth_first_search(farmer_problem)

print("\n--- Farmer Crosses River Solution ---")
for step, state in enumerate(solution):
    # State format: (Farmer, Wolf, Chicken, Grain)
    print(f"Step {step}: {state}")


--- Farmer Crosses River Solution ---
Step 0: (0, 0, 0, 0)
Step 1: (1, 0, 1, 0)
Step 2: (0, 0, 1, 0)
Step 3: (1, 1, 1, 0)
Step 4: (0, 1, 0, 0)
Step 5: (1, 1, 0, 1)
Step 6: (0, 1, 0, 1)
Step 7: (1, 1, 1, 1)
